Bonus Code for Chapter 5
Alternative Weight Loading from Hugging Face Model Hub using Transformers

In [ ]:
# 从 importlib.metadata 导入 version 函数,用于查询已安装第三方库的版本号
from importlib.metadata import version

# 列出本notebook依赖的关键库:
# numpy        -> 用于权重张量的拆分(np.split)等数值运算
# torch        -> PyTorch深度学习框架,构建/运行自定义GPT模型
# transformers -> Hugging Face库,用于下载并加载官方预训练的GPT-2权重
pkgs = ["numpy", "torch", "transformers"]
for p in pkgs:
    # 打印每个库的版本号,便于排查环境兼容性问题
    print(f"{p} version: {version(p)}")

In [ ]:
# 从Hugging Face transformers库导入GPT2Model类
# GPT2Model是HF官方实现的GPT-2骨干网络(不含语言模型头),可直接加载OpenAI发布的预训练权重
from transformers import GPT2Model


# allowed model names
# 这里定义了本书自定义配置名称 -> Hugging Face Hub上对应模型仓库ID的映射
# 例如 "gpt2-small (124M)" 对应 Hugging Face Hub上的 "openai-community/gpt2"
model_names = {
    "gpt2-small (124M)": "openai-community/gpt2",
    "gpt2-medium (355M)": "openai-community/gpt2-medium",
    "gpt2-large (774M)": "openai-community/gpt2-large",
    "gpt2-xl (1558M)": "openai-community/gpt2-xl"
}

# 选择要加载的模型规格,这里使用最小的124M参数版本
CHOOSE_MODEL = "gpt2-small (124M)"

# 调用from_pretrained从Hugging Face Hub下载(或使用本地缓存)对应的预训练权重
# cache_dir="checkpoints" 指定权重文件的本地缓存目录,避免重复下载
gpt_hf = GPT2Model.from_pretrained(model_names[CHOOSE_MODEL], cache_dir="checkpoints")
# 切换为eval模式,关闭dropout等训练专用行为(推理/权重迁移时的标准做法)
gpt_hf.eval()

In [ ]:
# 基础配置字典,后续会根据所选模型规格(model_configs)进行更新补全
BASE_CONFIG = {
    "vocab_size": 50257,    # Vocabulary size 词表大小(GPT-2使用的BPE词表)
    "context_length": 1024, # Context length 上下文长度(最大支持的序列长度)
    "drop_rate": 0.0,       # Dropout rate dropout比例,这里设为0(推理阶段不需要正则化)
    "qkv_bias": True        # Query-key-value bias 是否在Q/K/V线性层中使用偏置项(GPT-2原始实现使用了bias)
}

# 不同规格GPT-2模型对应的结构超参数:
# emb_dim  -> 词嵌入/隐藏层维度
# n_layers -> Transformer块(层)的数量
# n_heads  -> 多头注意力的头数
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}


# 根据前面CHOOSE_MODEL选定的规格,将对应的结构超参数合并进BASE_CONFIG
# 合并后BASE_CONFIG中就同时包含通用配置与该规格特有的emb_dim/n_layers/n_heads
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [ ]:
# 辅助函数:在把Hugging Face权重赋值给自定义模型参数前,先校验张量形状是否一致
# 这样可以尽早发现权重映射写错(例如转置漏了/维度对不上)的问题,而不是等到前向传播才报错
def assign_check(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    # 用right(来自HF预训练模型)的数值克隆一份新的Tensor,并包装成torch.nn.Parameter返回
    # clone().detach() 确保新参数与原始HF模型的计算图/梯度历史无关联
    return torch.nn.Parameter(right.clone().detach())

In [ ]:
import numpy as np  # 用于将HF的合并QKV权重(c_attn)按最后一维拆分成Q/K/V三部分


def load_weights(gpt, gpt_hf):
    # 说明:
    # HF的GPT2Model内部使用的是Conv1D层(权重形状为[in_features, out_features]),
    # 与PyTorch nn.Linear的权重形状[out_features, in_features]互为转置关系,
    # 因此下面凡是从HF权重赋值给nn.Linear权重的地方都需要做 .T 转置。

    # 取出Hugging Face预训练模型的完整参数字典(键为HF内部命名,如"wte.weight"等)
    d = gpt_hf.state_dict()

    # 位置编码矩阵(wpe = word position embedding)直接对应我们模型的pos_emb
    gpt.pos_emb.weight = assign_check(gpt.pos_emb.weight, d["wpe.weight"])
    # 词嵌入矩阵(wte = word token embedding)直接对应我们模型的tok_emb
    gpt.tok_emb.weight = assign_check(gpt.tok_emb.weight, d["wte.weight"])

    # 逐层(逐个Transformer Block)搬运权重
    for b in range(BASE_CONFIG["n_layers"]):
        # HF把Q、K、V的权重合并存放在同一个c_attn.weight张量里(沿最后一维拼接)
        # 这里用np.split把它拆回三份,分别对应Query/Key/Value的权重
        q_w, k_w, v_w = np.split(d[f"h.{b}.attn.c_attn.weight"], 3, axis=-1)
        # 拆分出的权重是Conv1D格式,需要转置(.T)后才能赋值给nn.Linear的weight
        gpt.trf_blocks[b].att.W_query.weight = assign_check(gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign_check(gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign_check(gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        # 同样地,c_attn.bias是Q/K/V三个偏置项拼接在一起的,需要拆分
        q_b, k_b, v_b = np.split(d[f"h.{b}.attn.c_attn.bias"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign_check(gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign_check(gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign_check(gpt.trf_blocks[b].att.W_value.bias, v_b)


        # 注意力输出投影层(c_proj)权重同样需要转置后再赋值
        gpt.trf_blocks[b].att.out_proj.weight = assign_check(gpt.trf_blocks[b].att.out_proj.weight, d[f"h.{b}.attn.c_proj.weight"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign_check(gpt.trf_blocks[b].att.out_proj.bias, d[f"h.{b}.attn.c_proj.bias"])

        # 前馈网络(MLP/FeedForward)第一层:c_fc对应升维的全连接层
        gpt.trf_blocks[b].ff.layers[0].weight = assign_check(gpt.trf_blocks[b].ff.layers[0].weight, d[f"h.{b}.mlp.c_fc.weight"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign_check(gpt.trf_blocks[b].ff.layers[0].bias, d[f"h.{b}.mlp.c_fc.bias"])
        # 前馈网络第二层:c_proj对应降维回embedding维度的全连接层
        gpt.trf_blocks[b].ff.layers[2].weight = assign_check(gpt.trf_blocks[b].ff.layers[2].weight, d[f"h.{b}.mlp.c_proj.weight"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign_check(gpt.trf_blocks[b].ff.layers[2].bias, d[f"h.{b}.mlp.c_proj.bias"])

        # 两个LayerNorm层(norm1对应注意力前的ln_1,norm2对应FFN前的ln_2)
        # 这里的LayerNorm权重/偏置无需转置,直接赋值即可
        gpt.trf_blocks[b].norm1.scale = assign_check(gpt.trf_blocks[b].norm1.scale, d[f"h.{b}.ln_1.weight"])
        gpt.trf_blocks[b].norm1.shift = assign_check(gpt.trf_blocks[b].norm1.shift, d[f"h.{b}.ln_1.bias"])
        gpt.trf_blocks[b].norm2.scale = assign_check(gpt.trf_blocks[b].norm2.scale, d[f"h.{b}.ln_2.weight"])
        gpt.trf_blocks[b].norm2.shift = assign_check(gpt.trf_blocks[b].norm2.shift, d[f"h.{b}.ln_2.bias"])

        # 注意:以下三行(最终LayerNorm与输出头)逻辑上只需要在for循环外执行一次即可,
        # 但原始代码把它们保留在了循环体内,导致每次迭代都重复赋值同样的值。
        # 这不会导致最终结果出错(赋的值始终相同),只是存在多余的重复计算;
        # 这里保留原始执行逻辑不做改动,仅作标注说明。
        gpt.final_norm.scale = assign_check(gpt.final_norm.scale, d["ln_f.weight"])
        gpt.final_norm.shift = assign_check(gpt.final_norm.shift, d["ln_f.bias"])
        # 权重绑定(weight tying):输出头(语言模型头)复用词嵌入矩阵wte.weight,而不是单独训练一套参数
        gpt.out_head.weight = assign_check(gpt.out_head.weight, d["wte.weight"])

In [ ]:
import torch
# [bug修正] 原代码此处为 `from Build_an_LLM_from_Scratch.ch04 import GPTModel`,
# 这是误写的本地路径式导入,实际可安装的包名是 llms_from_scratch,这里修正为正确的导入路径
from llms_from_scratch.ch04 import GPTModel
# For llms_from_scratch installation instructions, see:
# https://github.com/rasbt/LLMs-from-scratch/tree/main/


# 用前面构建好的BASE_CONFIG实例化一个我们自己实现的GPTModel(此时参数是随机初始化的)
gpt = GPTModel(BASE_CONFIG)

# 优先使用GPU(cuda),否则退回CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 调用前面定义的load_weights函数,把Hugging Face预训练权重逐一映射并赋值到gpt模型中
load_weights(gpt, gpt_hf)

In [ ]:
import tiktoken
# [bug修正] 原代码此处为 `from Build_an_LLM_from_Scratch.ch05 import ...`,
# 同样是误写的本地路径式导入,这里修正为正确的包名 llms_from_scratch
from llms_from_scratch.ch05 import generate, text_to_token_ids, token_ids_to_text


# 固定随机种子,保证文本生成过程可复现
torch.manual_seed(123)

# 使用GPT-2官方的BPE分词器(tiktoken实现),与预训练权重保持一致
tokenizer = tiktoken.get_encoding("gpt2")

# 用加载好预训练权重的模型进行自回归文本生成,验证权重迁移是否成功
token_ids = generate(
    model=gpt.to(device),                                              # 将模型移动到目标设备(GPU/CPU)
    idx=text_to_token_ids("Every effort moves", tokenizer).to(device),  # 将起始提示词编码为token id
    max_new_tokens=30,                                                  # 最多生成30个新token
    context_size=BASE_CONFIG["context_length"],                        # 模型支持的最大上下文长度
    top_k=1,                                                            # top-k采样,k=1即贪心解码(始终取概率最高的token)
    temperature=1.0                                                     # 采样温度,1.0表示不额外缩放概率分布
)

# 将生成的token id解码回可读文本并打印
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))